# Causal Impact Analysis: Max Reach Campaigns

## Overview

**Objective:** Measure the incremental lift of the Jaguar release on advertising KPIs using **Bayesian Structural Time Series** with synthetic control methodology.

**Approach:**
1. Pull daily KPI data from coredw (pre and post release)
2. Cluster advertisers by behavioral patterns (spend, volatility, efficiency)
3. Run CausalImpact analysis per cluster to detect effects
4. Compare results across clusters to identify responders vs non-responders

**Key Metrics:** IVR, CVR, VVR, CPA, CPV, ROAS, AOV

---
## 1. Setup & Imports

In [ ]:
# =============================================================================
# INSTALL DEPENDENCIES (run once per cluster)
# =============================================================================

%pip install pycausalimpact --quiet

In [ ]:
# =============================================================================
# IMPORTS AND SETUP
# =============================================================================

# Standard library
import warnings
import os
import json
import requests
from typing import List, Optional, Dict, Any, Tuple
from datetime import datetime, timedelta, date

# Suppress noisy warnings
warnings.filterwarnings("ignore", message=".*threadpoolctl.*")
warnings.filterwarnings("ignore", message=".*DecimalType.*")
warnings.filterwarnings("ignore", category=UserWarning, module="pyspark")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*DataFrame.fillna.*")
os.environ["OMP_NUM_THREADS"] = "1"  # Reduce threadpool noise

# Data manipulation
import pandas as pd
import numpy as np
pd.options.mode.chained_assignment = None  # Suppress SettingWithCopyWarning

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Statistics
from scipy import stats

# Spark
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Google auth (for Vault/coredw access)
from google.auth.transport import requests as g_request
from google.auth import compute_engine

# Machine Learning (clustering)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Causal Impact
from causalimpact import CausalImpact


print("Imports loaded successfully")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")


---
## 2. Configuration Parameters

In [ ]:
# =============================================================================
# ANALYSIS PARAMETERS
# =============================================================================

# Release date (intervention date)
RELEASE_DATE = date(2025, 11, 19)

# Vertical filter
# Options:
#   Single vertical:    VERTICAL_IDS = [129000]
#   Multiple verticals: VERTICAL_IDS = [129000, 130000, 131000]
#   All verticals:      VERTICAL_IDS = None
VERTICAL_IDS: Optional[List[int]] = None  # Analyze all verticals

# Primary metric
PRIMARY_METRIC = "ivr"

# =============================================================================
# PERIOD CONFIGURATION (WEEKLY)
# =============================================================================

# How many weeks of pre-period data to use (longer = better baseline)
PRE_PERIOD_WEEKS = 52  # 1 year

# Post-period: use all complete weeks since release
days_since_release = (date.today() - timedelta(days=1) - RELEASE_DATE).days
POST_PERIOD_WEEKS = days_since_release // 7

# Calculate date boundaries
PRE_START = RELEASE_DATE - timedelta(weeks=PRE_PERIOD_WEEKS)
PRE_END = RELEASE_DATE - timedelta(days=1)
POST_START = RELEASE_DATE + timedelta(days=1)
POST_END = RELEASE_DATE + timedelta(weeks=POST_PERIOD_WEEKS)

print(f"Release Date:       {RELEASE_DATE}")
print(f"Pre-period:         {PRE_START} to {PRE_END} ({PRE_PERIOD_WEEKS} weeks)")
print(f"Post-period:        {POST_START} to {POST_END} ({POST_PERIOD_WEEKS} weeks)")
print(f"Vertical IDs:       {VERTICAL_IDS if VERTICAL_IDS else 'All verticals'}")

---
## 3. Helper Functions

In [ ]:
# =============================================================================
# VAULT & DATABASE HELPERS
# =============================================================================

def token_for_url(url: str) -> str:
    """
    Get GCP identity token for the specified URL.
    Used for Vault authentication via workload identity.
    """
    request = g_request.Request()
    credentials = compute_engine.IDTokenCredentials(
        request=request,
        target_audience=url,
        use_metadata_identity_endpoint=True,
    )
    credentials.refresh(request)
    return credentials.token


def get_secret(secret_name: str) -> Dict:

    """
    Retrieve secret from Vault using GCP workload identity authentication.
    """
    vault_address = "https://vault.prod.in.mountain.com"
    role = "gcp-workloads"
    path = "shared/global/ti"

    jwt = token_for_url(f"{vault_address}/vault/gcp-workloads")

    auth_resp = requests.post(
        f"{vault_address}/v1/auth/gcp/login",
        headers={"Content-Type": "application/json"},
        data=json.dumps({"role": role, "jwt": jwt}),
    )
    auth_resp.raise_for_status()
    vault_token = auth_resp.json()["auth"]["client_token"]

    secret_resp = requests.get(
        f"{vault_address}/v1/secret/data/{path}/{secret_name}",
        headers={"X-Vault-Token": vault_token},
    )
    secret_resp.raise_for_status()

    return secret_resp.json().get("data", {}).get("data")


def load_postgres_query(query: str, session: SparkSession) -> DataFrame:
    """
    Execute a query against coredw (Greenplum) and return results as Spark DataFrame.
    """
    secrets = get_secret("coredw")

    return (
        session.read
        .format("jdbc")
        .option("url", f"jdbc:postgresql://{secrets['hostname']}:{secrets['port']}/{secrets['database']}")
        .option("dbtable", query)
        .option("user", secrets["username"])
        .option("password", secrets["password"])
        .option("driver", "org.postgresql.Driver")
        .load()
    )


print("Helper functions loaded.")

---
## 4. Build Daily KPI Query

In [ ]:
# =============================================================================
# BUILD SQL QUERY FOR DAILY KPIs
# =============================================================================

# Build vertical filter clause
if VERTICAL_IDS:
    vertical_filter = f"and av.vertical_id in ({','.join(map(str, VERTICAL_IDS))})"
else:
    vertical_filter = "-- all verticals (no filter)"

DAILY_KPI_QUERY = f"""
(
    /* ------------------------------------------------------------------------
       Step 0: Get Max Reach campaign groups (threshold < 3333)
       ------------------------------------------------------------------------ */
    with max_reach_campaigns as (
        select distinct campaign_group_id
        from dso.household_score_thresholds
        where threshold < 3333
    )

    /* ------------------------------------------------------------------------
       Step 1: Get advertisers in target vertical(s)
       ------------------------------------------------------------------------ */
    , advertisers_in_vertical as (
        select distinct
            av.advertiser_id
          , av.vertical_id
          , av.vertical_name
        from fpa.advertiser_verticals av
        where 1 = 1
            and av.type = 1
            {vertical_filter}
    )

    /* ------------------------------------------------------------------------
       Step 2: Get Funnel 1 campaign groups (filtered to Max Reach)
       ------------------------------------------------------------------------ */
    , qualifying_campaigns as (
        select distinct
            a.advertiser_id
          , a.vertical_id
          , a.vertical_name
          , cg.campaign_group_id
        from advertisers_in_vertical a
        inner join public.campaign_groups cg
            on cg.advertiser_id = a.advertiser_id
        inner join campaign_groups_raw cgr
            on cgr.campaign_group_id = cg.campaign_group_id
        inner join max_reach_campaigns mrc
            on mrc.campaign_group_id = cg.campaign_group_id
        where 1 = 1
            and cgr.objective_id = 1
    )

    /* ------------------------------------------------------------------------
       Step 3: Identify last-touch attribution advertisers
       ------------------------------------------------------------------------ */
    , last_touch_advertisers as (
        select distinct advertiser_id
        from r2.advertiser_settings
        where reporting_style = 'last_touch'
    )

    /* ------------------------------------------------------------------------
       Step 4: Aggregate daily KPIs by vertical
       ------------------------------------------------------------------------ */
    select
        qc.vertical_id
      , qc.vertical_name
      , d.day

        -- Activity counts
      , count(distinct d.campaign_group_id) as active_campaigns
      , count(distinct qc.advertiser_id) as active_advertisers

        -- Volume metrics
      , sum(d.impressions) as impressions
      , sum(d.media_spend + d.data_spend + d.platform_spend)::float as spend

        -- Conversions (attribution-aware)
      , sum(
            case
                when lt.advertiser_id is null
                    then d.click_conversions + d.view_conversions + coalesce(d.competing_view_conversions, 0)
                else d.click_conversions + d.view_conversions
            end
        ) as conversions

        -- Order value (attribution-aware)
      , sum(
            case
                when lt.advertiser_id is null
                    then d.click_order_value + d.view_order_value + coalesce(d.competing_view_order_value, 0)
                else d.click_order_value + d.view_order_value
            end
        )::float as order_value

        -- Verified visits (attribution-aware)
      , sum(
            case
                when lt.advertiser_id is null
                    then d.clicks + d.views + coalesce(d.competing_views, 0)
                else d.clicks + d.views
            end
        ) as vv

        -- Uniques
      , sum(d.uniques) as uniques

    from qualifying_campaigns qc
    inner join summarydata.sum_by_campaign_group_by_day d
        on d.campaign_group_id = qc.campaign_group_id
    left join last_touch_advertisers lt
        on lt.advertiser_id = qc.advertiser_id
    where 1 = 1
        and d.day >= '{PRE_START}'::date
        and d.day <= '{POST_END}'::date
        and d.day <> '{RELEASE_DATE}'::date  -- exclude release date
        and d.impressions > 0
    group by 1, 2, 3
    order by 1, 3
) as daily_kpis
"""

print("Query built successfully.")
print(f"\nVertical filter: {vertical_filter}")
print(f"Date range: {PRE_START} to {POST_END} (excluding {RELEASE_DATE})")

In [ ]:
# =============================================================================
# BUILD SQL QUERY FOR DAILY KPIs (will aggregate to weekly after outlier removal)
# =============================================================================

DAILY_KPI_QUERY = f"""
(
    /* Get Max Reach campaign groups (threshold < 3333) */
    with max_reach_campaigns as (
        select distinct campaign_group_id
        from dso.household_score_thresholds
        where threshold < 3333
    )

    , advertisers_in_vertical as (
        select distinct
            av.advertiser_id
          , av.vertical_id
          , av.vertical_name
        from fpa.advertiser_verticals av
        where 1 = 1
            and av.type = 1
            {vertical_filter}
    )

    , qualifying_campaigns as (
        select distinct
            a.advertiser_id
          , a.vertical_id
          , a.vertical_name
          , cg.campaign_group_id
        from advertisers_in_vertical a
        inner join public.campaign_groups cg
            on cg.advertiser_id = a.advertiser_id
        inner join campaign_groups_raw cgr
            on cgr.campaign_group_id = cg.campaign_group_id
        inner join max_reach_campaigns mrc
            on mrc.campaign_group_id = cg.campaign_group_id
        where 1 = 1
            and cgr.objective_id = 1
    )

    , last_touch_advertisers as (
        select distinct advertiser_id
        from r2.advertiser_settings
        where reporting_style = 'last_touch'
    )

    /* Aggregate to DAILY level */
    select
        qc.vertical_id
      , qc.vertical_name
      , d.day

        -- Activity counts
      , count(distinct d.campaign_group_id) as active_campaigns
      , count(distinct qc.advertiser_id) as active_advertisers

        -- Volume metrics
      , sum(d.impressions) as impressions
      , sum(d.media_spend + d.data_spend + d.platform_spend)::float as spend

        -- Conversions (attribution-aware)
      , sum(
            case
                when lt.advertiser_id is null
                    then d.click_conversions + d.view_conversions + coalesce(d.competing_view_conversions, 0)
                else d.click_conversions + d.view_conversions
            end
        ) as conversions

        -- Order value (attribution-aware)
      , sum(
            case
                when lt.advertiser_id is null
                    then d.click_order_value + d.view_order_value + coalesce(d.competing_view_order_value, 0)
                else d.click_order_value + d.view_order_value
            end
        )::float as order_value

        -- Verified visits (attribution-aware)
      , sum(
            case
                when lt.advertiser_id is null
                    then d.clicks + d.views + coalesce(d.competing_views, 0)
                else d.clicks + d.views
            end
        ) as vv

        -- Uniques
      , sum(d.uniques) as uniques

    from qualifying_campaigns qc
    inner join summarydata.sum_by_campaign_group_by_day d
        on d.campaign_group_id = qc.campaign_group_id
    left join last_touch_advertisers lt
        on lt.advertiser_id = qc.advertiser_id
    where 1 = 1
        and d.day >= '{PRE_START}'::date
        and d.day <= '{POST_END}'::date
        and d.impressions > 0
    group by 1, 2, 3
    order by 1, 3
) as daily_kpis
"""

print("Daily KPI query built successfully.")
print(f"\nVertical filter: {vertical_filter}")
print(f"Date range: {PRE_START} to {POST_END}")

---
## 5. Load Data from coredw

In [ ]:
# =============================================================================
# LOAD DAILY KPI DATA
# =============================================================================

daily_kpis_df = load_postgres_query(DAILY_KPI_QUERY, spark)

# Convert to pandas
daily_kpis_pd = daily_kpis_df.toPandas()

# Convert day to datetime
daily_kpis_pd["day"] = pd.to_datetime(daily_kpis_pd["day"])

# Convert Decimal columns to float
numeric_cols = ["impressions", "spend", "conversions", "order_value", "vv", "uniques", "active_campaigns", "active_advertisers"]
for c in numeric_cols:
    if c in daily_kpis_pd.columns:
        daily_kpis_pd[c] = daily_kpis_pd[c].astype(float)

print(f"Loaded {len(daily_kpis_pd):,} daily records")
print(f"Verticals: {daily_kpis_pd['vertical_name'].nunique()}")
print(f"Date range: {daily_kpis_pd['day'].min().date()} to {daily_kpis_pd['day'].max().date()}")

daily_kpis_pd.head()

In [ ]:
# =============================================================================
# DAILY OUTLIER DETECTION & WEEKLY AGGREGATION
# =============================================================================
# 
# Process:
# 1. Detect outlier DAYS using IQR on daily covariates
# 2. Remove outlier days (with detailed report)
# 3. Aggregate cleaned daily data to weekly
# 4. Limit post-period to MAX_POST_WEEKS
#
# This is more surgical than removing entire weeks — if 1 day is bad,
# we only lose 1/7th of the week's data instead of the whole week.
# =============================================================================

# ------------------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------------------

MAX_POST_WEEKS = 8  # Maximum post-period weeks

# Hard minimum overrides (set to None to use IQR, or a number for hard minimum)
HARD_MINIMUM_OVERRIDES = {
    "impressions": None,         # e.g., 5000 for daily minimum
    "spend": None,
    "uniques": None,
    "active_advertisers": None,
    "active_campaigns": None,
}

COVARIATES_TO_CHECK = ["impressions", "spend", "uniques", "active_advertisers", "active_campaigns"]

# ------------------------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------------------------

def detect_outliers_iqr(series):
    """Calculate IQR bounds for a series."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = max(0, q1 - 1.5 * iqr)
    upper_bound = q3 + 1.5 * iqr
    return lower_bound, upper_bound

# ------------------------------------------------------------------------------
# STEP 1: DETECT OUTLIER DAYS
# ------------------------------------------------------------------------------

print("=" * 80)
print("DAILY OUTLIER DETECTION")
print("=" * 80)

release_date_ts = pd.Timestamp(RELEASE_DATE)

# Split into pre and post for separate analysis
pre_daily = daily_kpis_pd[daily_kpis_pd["day"] < release_date_ts].copy()
post_daily = daily_kpis_pd[daily_kpis_pd["day"] >= release_date_ts].copy()

all_outlier_records = []
days_to_remove = set()

for vertical in daily_kpis_pd["vertical_name"].unique():
    print(f"\n📊 Vertical: {vertical}")
    print("-" * 40)
    
    v_data = pre_daily[pre_daily["vertical_name"] == vertical].copy()
    
    if len(v_data) == 0:
        print("  No pre-period data for this vertical")
        continue
    
    for covariate in COVARIATES_TO_CHECK:
        if covariate not in v_data.columns:
            continue
        
        hard_min = HARD_MINIMUM_OVERRIDES.get(covariate)
        
        if hard_min is not None:
            lower_bound = hard_min
            upper_bound = float('inf')
            method = "HARD MIN"
        else:
            lower_bound, upper_bound = detect_outliers_iqr(v_data[covariate].dropna())
            method = "IQR"
        
        # Find outliers in pre-period
        for idx, row in v_data.iterrows():
            value = row[covariate]
            flag = None
            
            if pd.notna(value):
                if value < lower_bound:
                    flag = "LOW"
                elif value > upper_bound and upper_bound != float('inf'):
                    flag = "HIGH"
            
            if flag:
                days_to_remove.add((row["day"], row["vertical_name"]))
                all_outlier_records.append({
                    "day": row["day"],
                    "vertical_name": vertical,
                    "covariate": covariate,
                    "value": value,
                    "lower_bound": lower_bound,
                    "upper_bound": upper_bound if upper_bound != float('inf') else None,
                    "flag": flag,
                    "method": method,
                    "period": "PRE"
                })
        
        # Check post-period for flagging (but not removal)
        v_post = post_daily[post_daily["vertical_name"] == vertical]
        for idx, row in v_post.iterrows():
            value = row[covariate]
            flag = None
            
            if pd.notna(value):
                if value < lower_bound:
                    flag = "LOW"
                elif value > upper_bound and upper_bound != float('inf'):
                    flag = "HIGH"
            
            if flag:
                all_outlier_records.append({
                    "day": row["day"],
                    "vertical_name": vertical,
                    "covariate": covariate,
                    "value": value,
                    "lower_bound": lower_bound,
                    "upper_bound": upper_bound if upper_bound != float('inf') else None,
                    "flag": flag,
                    "method": method,
                    "period": "POST"
                })
        
        ub_str = f"{upper_bound:,.2f}" if upper_bound != float('inf') else "∞"
        print(f"  {covariate:20s}: [{lower_bound:>12,.2f} — {ub_str:>12s}] ({method})")

# ------------------------------------------------------------------------------
# STEP 2: REPORT AND REMOVE OUTLIER DAYS
# ------------------------------------------------------------------------------

if all_outlier_records:
    outlier_df = pd.DataFrame(all_outlier_records)
    
    pre_outliers = outlier_df[outlier_df["period"] == "PRE"]
    post_outliers = outlier_df[outlier_df["period"] == "POST"]
    
    if len(pre_outliers) > 0:
        print("\n" + "=" * 80)
        print("PRE-PERIOD OUTLIER DAYS (will be REMOVED before aggregation)")
        print("=" * 80)
        
        unique_pre_days = pre_outliers.groupby(["day", "vertical_name"])["covariate"].apply(list).reset_index()
        print(f"\n{len(unique_pre_days)} day(s) flagged for removal:\n")
        
        for _, row in pre_outliers.sort_values(["vertical_name", "day", "covariate"]).iterrows():
            ub_str = f"{row['upper_bound']:,.2f}" if row['upper_bound'] else "∞"
            print(f"  {row['day'].strftime('%Y-%m-%d')} | {row['vertical_name'][:25]:25s} | "
                  f"{row['covariate']:20s} | {row['value']:>14,.2f} | "
                  f"bounds: [{row['lower_bound']:>12,.2f} — {ub_str:>12s}] | {row['flag']:4s}")
    
    if len(post_outliers) > 0:
        print("\n" + "=" * 80)
        print("POST-PERIOD OUTLIER DAYS (flagged but KEPT)")
        print("=" * 80)
        
        unique_post_days = post_outliers.groupby(["day", "vertical_name"])["covariate"].apply(list).reset_index()
        print(f"\n⚠️  {len(unique_post_days)} day(s) with outliers (NOT removed):\n")
        
        for _, row in post_outliers.sort_values(["vertical_name", "day", "covariate"]).iterrows():
            ub_str = f"{row['upper_bound']:,.2f}" if row['upper_bound'] else "∞"
            print(f"  {row['day'].strftime('%Y-%m-%d')} | {row['vertical_name'][:25]:25s} | "
                  f"{row['covariate']:20s} | {row['value']:>14,.2f} | {row['flag']:4s}")
        print("\n  Post-period outliers kept to measure actual behavior.")
else:
    print("\n✅ No outlier days detected.")

# Remove outlier days from pre-period
pre_before = len(pre_daily)
for (day, vertical) in days_to_remove:
    pre_daily = pre_daily[~((pre_daily["day"] == day) & (pre_daily["vertical_name"] == vertical))]

print(f"\n✅ Removed {pre_before - len(pre_daily)} outlier day(s) from pre-period.")

# Recombine
daily_kpis_clean = pd.concat([pre_daily, post_daily], ignore_index=True)

# ------------------------------------------------------------------------------
# STEP 3: AGGREGATE CLEANED DAILY DATA TO WEEKLY
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("AGGREGATING CLEANED DAILY DATA TO WEEKLY")
print("=" * 80)

# Add week_start column
daily_kpis_clean["week_start"] = daily_kpis_clean["day"].dt.to_period("W").dt.start_time

# Aggregate to weekly
weekly_kpis_pd = daily_kpis_clean.groupby(["vertical_id", "vertical_name", "week_start"]).agg({
    "impressions": "sum",
    "spend": "sum",
    "conversions": "sum",
    "order_value": "sum",
    "vv": "sum",
    "uniques": "sum",
    "active_campaigns": "max",  # Use max for the week
    "active_advertisers": "max",  # Use max for the week
    "day": "count"  # Count days in this week (for transparency)
}).reset_index()

weekly_kpis_pd = weekly_kpis_pd.rename(columns={"day": "days_in_week"})

print(f"\nAggregated to {len(weekly_kpis_pd)} weekly records")
print(f"\nDays per week distribution:")
print(weekly_kpis_pd["days_in_week"].value_counts().sort_index().to_string())

# Flag weeks with missing days
incomplete_weeks = weekly_kpis_pd[weekly_kpis_pd["days_in_week"] < 7]
if len(incomplete_weeks) > 0:
    print(f"\n⚠️  {len(incomplete_weeks)} week(s) have fewer than 7 days due to outlier removal or data gaps:")
    print(incomplete_weeks[["week_start", "vertical_name", "days_in_week"]].to_string())

# ------------------------------------------------------------------------------
# STEP 4: LIMIT POST-PERIOD
# ------------------------------------------------------------------------------

pre_weekly = weekly_kpis_pd[weekly_kpis_pd["week_start"] < release_date_ts].copy()
post_weekly = weekly_kpis_pd[weekly_kpis_pd["week_start"] >= release_date_ts].copy()

# Limit post-period
post_weekly = post_weekly.sort_values("week_start").groupby("vertical_name").head(MAX_POST_WEEKS)

print(f"\n" + "=" * 80)
print("POST-PERIOD LIMITING")
print("=" * 80)
print(f"Post-period limited to {MAX_POST_WEEKS} weeks per vertical")

# Recombine
weekly_kpis_pd = pd.concat([pre_weekly, post_weekly], ignore_index=True)
weekly_kpis_pd = weekly_kpis_pd.sort_values(["vertical_name", "week_start"]).reset_index(drop=True)

# ------------------------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL DATA SUMMARY")
print("=" * 80)
print(f"Pre-period weeks:  {len(pre_weekly)}")
print(f"Post-period weeks: {len(post_weekly)}")
print(f"Total weeks:       {len(weekly_kpis_pd)}")
print("=" * 80)

In [ ]:
# =============================================================================
# CALCULATE EFFICIENCY METRICS
# =============================================================================

weekly_kpis_pd["ivr"] = weekly_kpis_pd["vv"] / weekly_kpis_pd["impressions"].replace(0, np.nan)
weekly_kpis_pd["cvr"] = weekly_kpis_pd["conversions"] / weekly_kpis_pd["vv"].replace(0, np.nan)
weekly_kpis_pd["vvr"] = weekly_kpis_pd["vv"] / weekly_kpis_pd["uniques"].replace(0, np.nan)
weekly_kpis_pd["cpa"] = weekly_kpis_pd["spend"] / weekly_kpis_pd["conversions"].replace(0, np.nan)
weekly_kpis_pd["cpv"] = weekly_kpis_pd["spend"] / weekly_kpis_pd["vv"].replace(0, np.nan)
weekly_kpis_pd["roas"] = weekly_kpis_pd["order_value"] / weekly_kpis_pd["spend"].replace(0, np.nan)
weekly_kpis_pd["aov"] = weekly_kpis_pd["order_value"] / weekly_kpis_pd["conversions"].replace(0, np.nan)

# Add lagged IVR
weekly_kpis_pd = weekly_kpis_pd.sort_values(["vertical_name", "week_start"])
weekly_kpis_pd["ivr_lag1"] = weekly_kpis_pd.groupby("vertical_name")["ivr"].shift(1)

print("Efficiency metrics calculated.")
print(f"\nSample data:")
weekly_kpis_pd[["week_start", "vertical_name", "impressions", "vv", "ivr", "days_in_week"]].head(10)

In [ ]:
# =============================================================================
# VALIDATION: CHECK DATA COMPLETENESS
# =============================================================================

release_date_dt = pd.Timestamp(RELEASE_DATE)

validation = weekly_kpis_pd.groupby("vertical_name").agg(
    pre_weeks=pd.NamedAgg(column="week_start", aggfunc=lambda x: (x < release_date_dt).sum()),
    post_weeks=pd.NamedAgg(column="week_start", aggfunc=lambda x: (x >= release_date_dt).sum()),
    total_impressions=pd.NamedAgg(column="impressions", aggfunc="sum"),
    avg_ivr=pd.NamedAgg(column="ivr", aggfunc="mean"),
).reset_index()

print("Data completeness by vertical:")
display(validation)

---
## 6. Causal Impact Analysis Functions

# Advertiser Clustering Module

This section clusters advertisers by **behavioral similarity** rather than pre-defined verticals.

**Why clustering?**
- Pre-defined verticals are arbitrary business categories
- Advertisers in the same vertical may behave very differently
- Clustering groups advertisers by actual behavioral patterns
- May produce better counterfactuals and reveal hidden effects

**Toggle:** Set `USE_CLUSTERING = True` to use clusters, `False` to use verticals.

In [ ]:
# =============================================================================
# ADVERTISER CLUSTERING MODULE
# =============================================================================
# Instead of using pre-defined verticals, cluster advertisers by behavioral
# similarity. This may produce better counterfactuals and reveal effects 
# masked by arbitrary vertical groupings.
#
# Toggle USE_CLUSTERING to switch between:
#   True  = Cluster-based analysis (behavioral groupings)
#   False = Vertical-based analysis (original approach)
# =============================================================================

USE_CLUSTERING = True  # Set to False to use vertical-based analysis

# Clustering parameters
MIN_WEEKS_FOR_CLUSTERING = 12  # Minimum weeks of data required per advertiser
MAX_CLUSTERS = 15              # Maximum clusters to test in elbow method
RANDOM_STATE = 42              # For reproducibility

# Imports for clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("CLUSTERING MODULE")
print("=" * 80)
print(f"USE_CLUSTERING: {USE_CLUSTERING}")
if USE_CLUSTERING:
    print(f"MIN_WEEKS_FOR_CLUSTERING: {MIN_WEEKS_FOR_CLUSTERING}")
    print(f"MAX_CLUSTERS: {MAX_CLUSTERS}")
else:
    print("Clustering disabled - will use vertical-based analysis")

In [ ]:
# =============================================================================
# STEP 1: LOAD ADVERTISER-LEVEL DATA FOR CLUSTERING
# =============================================================================

if USE_CLUSTERING:
    print("\n" + "=" * 80)
    print("LOADING ADVERTISER-LEVEL DATA")
    print("=" * 80)
    
    # Query advertiser-level weekly metrics (pre-period only for feature engineering)
    ADVERTISER_CLUSTERING_QUERY = f"""
    (
        with max_reach_campaigns as (
            select distinct campaign_group_id
            from dso.household_score_thresholds
            where threshold < 3333
        )
        
        , advertisers_in_vertical as (
            select distinct
                av.advertiser_id
              , av.vertical_id
              , av.vertical_name
            from fpa.advertiser_verticals av
            where 1 = 1
                and av.type = 1
                {vertical_filter}
        )
        
        , qualifying_campaigns as (
            select distinct
                a.advertiser_id
              , a.vertical_id
              , a.vertical_name
              , cg.campaign_group_id
            from advertisers_in_vertical a
            inner join public.campaign_groups cg
                on cg.advertiser_id = a.advertiser_id
            inner join campaign_groups_raw cgr
                on cgr.campaign_group_id = cg.campaign_group_id
            inner join max_reach_campaigns mrc
                on mrc.campaign_group_id = cg.campaign_group_id
            where cgr.objective_id = 1
        )
        
        , last_touch_advertisers as (
            select distinct advertiser_id
            from r2.advertiser_settings
            where reporting_style = 'last_touch'
        )
        
        select
            qc.advertiser_id
          , qc.vertical_id
          , qc.vertical_name
          , date_trunc('week', d.day)::date as week_start
          , count(distinct d.campaign_group_id) as active_campaigns
          , sum(d.impressions) as impressions
          , sum(d.media_spend + d.data_spend + d.platform_spend)::float as spend
          , sum(
                case when lt.advertiser_id is null
                    then d.click_conversions + d.view_conversions + coalesce(d.competing_view_conversions, 0)
                    else d.click_conversions + d.view_conversions
                end
            ) as conversions
          , sum(
                case when lt.advertiser_id is null
                    then d.clicks + d.views + coalesce(d.competing_views, 0)
                    else d.clicks + d.views
                end
            ) as vv
          , sum(d.uniques) as uniques
        from qualifying_campaigns qc
        inner join summarydata.sum_by_campaign_group_by_day d
            on d.campaign_group_id = qc.campaign_group_id
        left join last_touch_advertisers lt
            on lt.advertiser_id = qc.advertiser_id
        where d.day >= '{PRE_START}'::date
            and d.day < '{RELEASE_DATE}'::date
            and d.impressions > 0
        group by 1, 2, 3, 4
        order by 1, 4
    ) as adv_weekly_pre
    """
    
    print("Loading advertiser weekly data (pre-period)...")
    adv_weekly_df = load_postgres_query(ADVERTISER_CLUSTERING_QUERY, spark)
    adv_weekly_pd = adv_weekly_df.toPandas()
    
    # Convert types
    adv_weekly_pd["week_start"] = pd.to_datetime(adv_weekly_pd["week_start"])
    for col in ["impressions", "spend", "conversions", "vv", "uniques", "active_campaigns"]:
        adv_weekly_pd[col] = adv_weekly_pd[col].astype(float)
    
    # Calculate IVR
    adv_weekly_pd["ivr"] = adv_weekly_pd["vv"] / adv_weekly_pd["impressions"].replace(0, np.nan)
    
    print(f"\nLoaded {len(adv_weekly_pd):,} advertiser-week records")
    print(f"Unique advertisers: {adv_weekly_pd['advertiser_id'].nunique():,}")
    print(f"Verticals represented: {adv_weekly_pd['vertical_name'].nunique()}")

In [ ]:
# =============================================================================
# STEP 2: ENGINEER BEHAVIORAL FEATURES PER ADVERTISER
# =============================================================================

if USE_CLUSTERING:
    print("\n" + "=" * 80)
    print("ENGINEERING ADVERTISER FEATURES")
    print("=" * 80)
    
    def calc_trend(series):
        """Calculate linear trend slope."""
        if len(series) < 3:
            return 0.0
        x = np.arange(len(series))
        try:
            slope, _, _, _, _ = stats.linregress(x, series.fillna(0))
            return slope
        except:
            return 0.0
    
    def calc_volatility(series):
        """Calculate coefficient of variation (volatility)."""
        if len(series) < 3 or series.mean() == 0:
            return 0.0
        return series.std() / series.mean()
    
    # Filter to advertisers with sufficient data
    adv_week_counts = adv_weekly_pd.groupby("advertiser_id")["week_start"].nunique()
    valid_advertisers = adv_week_counts[adv_week_counts >= MIN_WEEKS_FOR_CLUSTERING].index.tolist()
    
    print(f"Advertisers with >= {MIN_WEEKS_FOR_CLUSTERING} weeks: {len(valid_advertisers):,}")
    
    adv_data = adv_weekly_pd[adv_weekly_pd["advertiser_id"].isin(valid_advertisers)].copy()
    
    # Aggregate features per advertiser
    print("Calculating features...")
    
    adv_features = adv_data.groupby("advertiser_id").agg(
        # Volume metrics
        impressions_mean=("impressions", "mean"),
        impressions_std=("impressions", "std"),
        spend_mean=("spend", "mean"),
        uniques_mean=("uniques", "mean"),
        
        # Efficiency metrics
        ivr_mean=("ivr", "mean"),
        ivr_std=("ivr", "std"),
        
        # Activity metrics
        campaigns_mean=("active_campaigns", "mean"),
        weeks_active=("week_start", "nunique"),
        
        # For trend/volatility calculation
        impressions_series=("impressions", list),
        ivr_series=("ivr", list),
        
        # Keep vertical info
        vertical_name=("vertical_name", "first"),
        vertical_id=("vertical_id", "first"),
    ).reset_index()
    
    # Calculate trend and volatility
    adv_features["impressions_trend"] = adv_features["impressions_series"].apply(
        lambda x: calc_trend(pd.Series(x))
    )
    adv_features["ivr_trend"] = adv_features["ivr_series"].apply(
        lambda x: calc_trend(pd.Series(x))
    )
    adv_features["impressions_volatility"] = adv_features["impressions_series"].apply(
        lambda x: calc_volatility(pd.Series(x))
    )
    adv_features["ivr_volatility"] = adv_features["ivr_series"].apply(
        lambda x: calc_volatility(pd.Series(x).dropna())
    )
    
    # Drop series columns
    adv_features = adv_features.drop(columns=["impressions_series", "ivr_series"])
    
    # Remove extreme outliers before clustering (IVR > 100% is data quality issue)
    n_before = len(adv_features)
    adv_features = adv_features[adv_features["ivr_mean"] < 1.0]  # IVR should be < 100%
    adv_features = adv_features[adv_features["ivr_mean"] > 0]    # IVR should be positive
    n_removed = n_before - len(adv_features)
    if n_removed > 0:
        print(f"Removed {n_removed} advertisers with invalid IVR (>100% or <=0)")
    
    # Define feature columns for clustering
    FEATURE_COLS = [
        "impressions_mean", "impressions_std", "spend_mean", "uniques_mean",
        "ivr_mean", "ivr_std", "campaigns_mean", "weeks_active",
        "impressions_trend", "ivr_trend", "impressions_volatility", "ivr_volatility"
    ]
    
    # Fill NaN with 0
    adv_features[FEATURE_COLS] = adv_features[FEATURE_COLS].fillna(0)
    
    print(f"\nEngineered features for {len(adv_features):,} advertisers")
    print(f"\nFeature summary:")
    display(adv_features[FEATURE_COLS].describe().T[["mean", "std", "min", "max"]].round(4))

In [ ]:
# =============================================================================
# STEP 3: ELBOW METHOD - FIND OPTIMAL NUMBER OF CLUSTERS
# =============================================================================

if USE_CLUSTERING:
    print("\n" + "=" * 80)
    print("ELBOW METHOD FOR OPTIMAL K")
    print("=" * 80)
    
    # Log-transform skewed volume features to prevent scale-based clustering
    # This compresses large differences and lets behavioral patterns emerge
    LOG_TRANSFORM_COLS = ["impressions_mean", "impressions_std", "spend_mean", "uniques_mean"]
    
    adv_features_transformed = adv_features.copy()
    for col in LOG_TRANSFORM_COLS:
        if col in FEATURE_COLS:
            adv_features_transformed[col] = np.log1p(adv_features_transformed[col])
    
    print("Applied log1p transformation to:", LOG_TRANSFORM_COLS)
    
    # Prepare and scale features
    X = adv_features_transformed[FEATURE_COLS].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Test range of k values
    K_range = range(2, min(MAX_CLUSTERS + 1, len(adv_features) // 5))
    inertias = []
    silhouette_scores = []
    
    from sklearn.metrics import silhouette_score
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = kmeans.fit_predict(X_scaled)
        inertias.append(kmeans.inertia_)
        if k > 1:
            silhouette_scores.append(silhouette_score(X_scaled, labels))
        else:
            silhouette_scores.append(0)
    
    # Plot elbow curve and silhouette scores
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Elbow curve
    axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
    axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
    axes[0].set_ylabel('Inertia', fontsize=12)
    axes[0].set_title('Elbow Method', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    
    # Silhouette scores
    axes[1].plot(list(K_range), silhouette_scores, 'go-', linewidth=2, markersize=8)
    axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
    axes[1].set_ylabel('Silhouette Score', fontsize=12)
    axes[1].set_title('Silhouette Analysis', fontsize=14)
    axes[1].grid(True, alpha=0.3)
    
    # Find elbow using second derivative
    if len(inertias) > 2:
        diffs = np.diff(inertias)
        diffs2 = np.diff(diffs)
        elbow_idx = np.argmax(np.abs(diffs2)) + 2
        suggested_k = list(K_range)[min(elbow_idx, len(K_range)-1)]
    else:
        suggested_k = 5
    
    # Also consider silhouette score peak
    best_silhouette_k = list(K_range)[np.argmax(silhouette_scores)]
    
    axes[0].axvline(x=suggested_k, color='r', linestyle='--', label=f'Elbow: k={suggested_k}')
    axes[1].axvline(x=best_silhouette_k, color='r', linestyle='--', label=f'Best silhouette: k={best_silhouette_k}')
    
    axes[0].legend()
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nElbow method suggests: k = {suggested_k}")
    print(f"Best silhouette score at: k = {best_silhouette_k} (score: {max(silhouette_scores):.3f})")
    
    # For advertising segmentation, we want MORE clusters to capture behavioral diversity
    # Silhouette often suggests k=2-3 which is too coarse for targeting purposes
    MIN_CLUSTERS_FOR_TARGETING = 10  # Minimum useful segments for advertising
    MAX_CLUSTERS_FOR_TARGETING = 14  # Avoid over-fragmentation
    
    # Use elbow point but enforce minimum
    recommended_k = max(suggested_k, MIN_CLUSTERS_FOR_TARGETING)
    recommended_k = min(recommended_k, MAX_CLUSTERS_FOR_TARGETING)
    
    print(f"\n>>> RECOMMENDED: k = {recommended_k} (optimized for advertising segmentation) <<<")
    print(f"    Range: {MIN_CLUSTERS_FOR_TARGETING}-{MAX_CLUSTERS_FOR_TARGETING} clusters for targeting use cases")
    print("    Adjust CHOSEN_K in the next cell if needed.")

In [ ]:
# =============================================================================
# STEP 4: APPLY CLUSTERING
# =============================================================================

if USE_CLUSTERING:
    # =========================================================================
    # SET YOUR CHOSEN K HERE (based on elbow plot above)
    # =========================================================================
    CHOSEN_K = recommended_k  # Uses recommended value; modify if you prefer different k
    # =========================================================================
    
    print("=" * 80)
    print(f"APPLYING K-MEANS CLUSTERING (k={CHOSEN_K})")
    print("=" * 80)
    
    # Fit final model using transformed/scaled features
    kmeans_final = KMeans(n_clusters=CHOSEN_K, random_state=RANDOM_STATE, n_init=10)
    adv_features["cluster"] = kmeans_final.fit_predict(X_scaled)
    
    # Generate descriptive cluster labels using ORIGINAL (non-transformed) values
    cluster_profiles = adv_features.groupby("cluster").agg({
        "advertiser_id": "count",
        "impressions_mean": "mean",
        "spend_mean": "mean",
        "ivr_mean": "mean",
        "impressions_trend": "mean",
        "ivr_volatility": "mean",
        "vertical_name": lambda x: x.value_counts().head(2).index.tolist()
    }).round(4)
    
    cluster_profiles.columns = ["n_advertisers", "avg_impressions", "avg_spend", "avg_ivr", "avg_trend", "avg_ivr_volatility", "top_verticals"]
    
    # Create labels using percentile-based thresholds
    p33_impressions = adv_features["impressions_mean"].quantile(0.33)
    p66_impressions = adv_features["impressions_mean"].quantile(0.66)
    p50_ivr = adv_features["ivr_mean"].quantile(0.50)
    p50_volatility = adv_features["ivr_volatility"].quantile(0.50)
    
    cluster_labels = {}
    for c in range(CHOSEN_K):
        profile = cluster_profiles.loc[c]
        
        # Volume tier (3 levels)
        if profile["avg_impressions"] > p66_impressions:
            size = "Large"
        elif profile["avg_impressions"] > p33_impressions:
            size = "Mid"
        else:
            size = "Small"
        
        # Efficiency
        efficiency = "Hi-IVR" if profile["avg_ivr"] > p50_ivr else "Lo-IVR"
        
        # Trend
        trend = "↑" if profile["avg_trend"] > 0 else "↓"
        
        # Stability
        stability = "Volatile" if profile["avg_ivr_volatility"] > p50_volatility else "Stable"
        
        cluster_labels[c] = f"C{c}: {size} {efficiency} {stability} {trend}"
    
    adv_features["cluster_label"] = adv_features["cluster"].map(cluster_labels)
    
    print("\nCluster Summary:")
    print(cluster_profiles.to_string())
    
    print("\n\nCluster Labels:")
    for c, label in cluster_labels.items():
        n = cluster_profiles.loc[c, "n_advertisers"]
        pct = 100 * n / len(adv_features)
        print(f"  {label} (n={n}, {pct:.1f}%)")
    
    # Store mapping for later use
    advertiser_cluster_map = adv_features.set_index("advertiser_id")[["cluster", "cluster_label"]].to_dict()

In [ ]:
# =============================================================================
# STEP 5: VISUALIZE CLUSTERS
# =============================================================================

if USE_CLUSTERING:
    print("\n" + "=" * 80)
    print("CLUSTER VISUALIZATION")
    print("=" * 80)
    print("Note: PCA uses log-transformed volume features for balanced visualization")
    
    # PCA for 2D visualization (using log-transformed, scaled features)
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    adv_features["pca_1"] = X_pca[:, 0]
    adv_features["pca_2"] = X_pca[:, 1]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Scatter plot of clusters
    colors = plt.cm.tab10(np.linspace(0, 1, CHOSEN_K))
    for c in range(CHOSEN_K):
        mask = adv_features["cluster"] == c
        axes[0].scatter(
            adv_features.loc[mask, "pca_1"],
            adv_features.loc[mask, "pca_2"],
            c=[colors[c]],
            label=cluster_labels[c],
            alpha=0.6,
            s=50
        )
    
    axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
    axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
    axes[0].set_title("Advertiser Clusters (PCA Projection)")
    axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    
    # Heatmap of cluster centers (in log-transformed space)
    centers_original = scaler.inverse_transform(kmeans_final.cluster_centers_)
    centers_df = pd.DataFrame(centers_original, columns=FEATURE_COLS)
    
    # Normalize for heatmap
    centers_norm = (centers_df - centers_df.min()) / (centers_df.max() - centers_df.min() + 1e-10)
    
    im = axes[1].imshow(centers_norm.T, aspect='auto', cmap='RdYlGn')
    axes[1].set_xticks(range(CHOSEN_K))
    axes[1].set_xticklabels([f"C{i}" for i in range(CHOSEN_K)], fontsize=10)
    axes[1].set_yticks(range(len(FEATURE_COLS)))
    axes[1].set_yticklabels(FEATURE_COLS, fontsize=9)
    axes[1].set_title("Cluster Centers (Log-Transformed Features)")
    plt.colorbar(im, ax=axes[1], shrink=0.8)
    
    plt.tight_layout()
    plt.show()
    
    # Vertical composition per cluster
    print("\nVertical composition by cluster:")
    vert_cluster = adv_features.groupby(["cluster", "vertical_name"]).size().unstack(fill_value=0)
    display(vert_cluster)

In [ ]:
# =============================================================================
# STEP 6: LOAD FULL DATA AND MAP CLUSTERS
# =============================================================================

if USE_CLUSTERING:
    print("\n" + "=" * 80)
    print("LOADING FULL DATA WITH CLUSTER MAPPING")
    print("=" * 80)
    
    # Load full daily data (pre + post period) at advertiser level
    ADVERTISER_FULL_QUERY = f"""
    (
        with max_reach_campaigns as (
            select distinct campaign_group_id
            from dso.household_score_thresholds
            where threshold < 3333
        )
        
        , advertisers_in_vertical as (
            select distinct av.advertiser_id, av.vertical_id, av.vertical_name
            from fpa.advertiser_verticals av
            where av.type = 1 {vertical_filter}
        )
        
        , qualifying_campaigns as (
            select distinct a.advertiser_id, a.vertical_id, a.vertical_name, cg.campaign_group_id
            from advertisers_in_vertical a
            inner join public.campaign_groups cg on cg.advertiser_id = a.advertiser_id
            inner join campaign_groups_raw cgr on cgr.campaign_group_id = cg.campaign_group_id
            inner join max_reach_campaigns mrc on mrc.campaign_group_id = cg.campaign_group_id
            where cgr.objective_id = 1
        )
        
        , last_touch_advertisers as (
            select distinct advertiser_id from r2.advertiser_settings where reporting_style = 'last_touch'
        )
        
        select
            qc.advertiser_id, qc.vertical_id, qc.vertical_name, d.day
          , count(distinct d.campaign_group_id) as active_campaigns
          , sum(d.impressions) as impressions
          , sum(d.media_spend + d.data_spend + d.platform_spend)::float as spend
          , sum(case when lt.advertiser_id is null
                then d.click_conversions + d.view_conversions + coalesce(d.competing_view_conversions, 0)
                else d.click_conversions + d.view_conversions end) as conversions
          , sum(case when lt.advertiser_id is null
                then d.click_order_value + d.view_order_value + coalesce(d.competing_view_order_value, 0)
                else d.click_order_value + d.view_order_value end)::float as order_value
          , sum(case when lt.advertiser_id is null
                then d.clicks + d.views + coalesce(d.competing_views, 0)
                else d.clicks + d.views end) as vv
          , sum(d.uniques) as uniques
        from qualifying_campaigns qc
        inner join summarydata.sum_by_campaign_group_by_day d on d.campaign_group_id = qc.campaign_group_id
        left join last_touch_advertisers lt on lt.advertiser_id = qc.advertiser_id
        where d.day >= '{PRE_START}'::date and d.day <= '{POST_END}'::date
            and d.day <> '{RELEASE_DATE}'::date and d.impressions > 0
        group by 1, 2, 3, 4
    ) as adv_daily_full
    """
    
    print("Loading full advertiser daily data...")
    adv_full_df = load_postgres_query(ADVERTISER_FULL_QUERY, spark)
    adv_full_pd = adv_full_df.toPandas()
    
    # Convert types
    adv_full_pd["day"] = pd.to_datetime(adv_full_pd["day"])
    for col in ["impressions", "spend", "conversions", "order_value", "vv", "uniques", "active_campaigns"]:
        adv_full_pd[col] = adv_full_pd[col].astype(float)
    
    # Map clusters
    adv_full_pd["cluster"] = adv_full_pd["advertiser_id"].map(advertiser_cluster_map["cluster"])
    adv_full_pd["cluster_label"] = adv_full_pd["advertiser_id"].map(advertiser_cluster_map["cluster_label"])
    
    # Keep only advertisers that were clustered
    adv_full_pd = adv_full_pd.dropna(subset=["cluster"])
    adv_full_pd["cluster"] = adv_full_pd["cluster"].astype(int)
    
    print(f"Loaded {len(adv_full_pd):,} daily records")
    print(f"Advertisers with cluster assignment: {adv_full_pd['advertiser_id'].nunique():,}")

In [ ]:
# =============================================================================
# STEP 7: AGGREGATE BY CLUSTER (WITH OUTLIER FILTERING)
# =============================================================================

if USE_CLUSTERING:
    print("\n" + "=" * 80)
    print("AGGREGATING DATA BY CLUSTER")
    print("=" * 80)
    
    release_ts = pd.Timestamp(RELEASE_DATE)
    
    # Aggregate daily to cluster level
    cluster_daily = adv_full_pd.groupby(["cluster", "cluster_label", "day"]).agg({
        "impressions": "sum",
        "spend": "sum",
        "conversions": "sum",
        "order_value": "sum",
        "vv": "sum",
        "uniques": "sum",
        "active_campaigns": "sum",
        "advertiser_id": "nunique"
    }).reset_index().rename(columns={"advertiser_id": "active_advertisers"})
    
    # Split pre/post
    pre_daily = cluster_daily[cluster_daily["day"] < release_ts].copy()
    post_daily = cluster_daily[cluster_daily["day"] >= release_ts].copy()
    
    # IQR outlier detection on pre-period
    print("\nDetecting outliers in cluster daily data...")
    CLUSTER_COVARIATES_CHECK = ["impressions", "spend", "uniques", "active_advertisers", "active_campaigns"]
    
    outlier_days = set()
    for cluster_id in pre_daily["cluster"].unique():
        cdata = pre_daily[pre_daily["cluster"] == cluster_id]
        for cov in CLUSTER_COVARIATES_CHECK:
            series = cdata[cov].dropna()
            if len(series) < 4:
                continue
            q1, q3 = series.quantile(0.25), series.quantile(0.75)
            iqr = q3 - q1
            lb, ub = max(0, q1 - 1.5*iqr), q3 + 1.5*iqr
            for idx, row in cdata.iterrows():
                if pd.notna(row[cov]) and (row[cov] < lb or row[cov] > ub):
                    outlier_days.add((row["day"], row["cluster"]))
    
    # Remove outliers from pre-period
    pre_before = len(pre_daily)
    for (day, cluster) in outlier_days:
        pre_daily = pre_daily[~((pre_daily["day"] == day) & (pre_daily["cluster"] == cluster))]
    
    print(f"Removed {pre_before - len(pre_daily)} outlier day(s) from pre-period")
    
    # Recombine and aggregate to weekly
    cluster_daily_clean = pd.concat([pre_daily, post_daily], ignore_index=True)
    cluster_daily_clean["week_start"] = cluster_daily_clean["day"].dt.to_period("W").dt.start_time
    
    cluster_weekly = cluster_daily_clean.groupby(["cluster", "cluster_label", "week_start"]).agg({
        "impressions": "sum", "spend": "sum", "conversions": "sum", "order_value": "sum",
        "vv": "sum", "uniques": "sum", "active_campaigns": "max", "active_advertisers": "max",
        "day": "count"
    }).reset_index().rename(columns={"day": "days_in_week"})
    
    # Limit post-period
    pre_weekly = cluster_weekly[cluster_weekly["week_start"] < release_ts]
    post_weekly = cluster_weekly[cluster_weekly["week_start"] >= release_ts]
    post_weekly = post_weekly.sort_values("week_start").groupby("cluster").head(MAX_POST_WEEKS)
    
    cluster_weekly = pd.concat([pre_weekly, post_weekly]).sort_values(["cluster", "week_start"]).reset_index(drop=True)
    
    # Calculate efficiency metrics
    cluster_weekly["ivr"] = cluster_weekly["vv"] / cluster_weekly["impressions"].replace(0, np.nan)
    cluster_weekly["cvr"] = cluster_weekly["conversions"] / cluster_weekly["vv"].replace(0, np.nan)
    cluster_weekly["vvr"] = cluster_weekly["vv"] / cluster_weekly["uniques"].replace(0, np.nan)
    cluster_weekly["cpa"] = cluster_weekly["spend"] / cluster_weekly["conversions"].replace(0, np.nan)
    cluster_weekly["cpv"] = cluster_weekly["spend"] / cluster_weekly["vv"].replace(0, np.nan)
    cluster_weekly["roas"] = cluster_weekly["order_value"] / cluster_weekly["spend"].replace(0, np.nan)
    cluster_weekly["aov"] = cluster_weekly["order_value"] / cluster_weekly["conversions"].replace(0, np.nan)
    
    print(f"\nFinal cluster weekly data: {len(cluster_weekly)} records")
    print(f"\nData per cluster:")
    summary = cluster_weekly.groupby("cluster_label").agg(
        pre_weeks=("week_start", lambda x: (x < release_ts).sum()),
        post_weeks=("week_start", lambda x: (x >= release_ts).sum()),
        total_impressions=("impressions", "sum")
    )
    display(summary)

In [ ]:
# =============================================================================
# STEP 8: RUN CAUSAL IMPACT BY CLUSTER
# =============================================================================
# NOTE: Define run_causal_impact here so clustering module is self-contained

def run_causal_impact_cluster(
    data: pd.DataFrame,
    metric: str,
    covariates: List[str],
    pre_period: list,
    post_period: list,
    vertical_name: str = "All"
) -> dict:
    """
    Run CausalImpact analysis on a single metric for cluster analysis.
    """
    columns = [metric] + covariates
    ci_data = data[columns].copy()
    ci_data = ci_data.dropna(subset=[metric])
    ci_data[covariates] = ci_data[covariates].ffill().bfill()
    ci_data = ci_data.astype(float)
    
    if len(ci_data) < 10:
        print(f"  ⚠️  Insufficient data for {vertical_name} ({len(ci_data)} rows)")
        return None
    
    try:
        ci = CausalImpact(ci_data, pre_period, post_period)
        inferences = ci.inferences
        post_mask = inferences.index > pre_period[1]
        post_inferences = inferences[post_mask]
        actual_post = ci.post_data.iloc[:, 0]
        
        actual_avg = actual_post.mean()
        predicted_avg = post_inferences["preds"].mean()
        abs_effect = post_inferences["point_effects"].mean()
        rel_effect = abs_effect / predicted_avg if predicted_avg != 0 else np.nan
        ci_lower = post_inferences["point_effects_lower"].mean()
        ci_upper = post_inferences["point_effects_upper"].mean()
        p_value = ci.p_value
        
        # Significance indicator
        sig = "✓" if p_value < 0.05 else ""
        direction = "↑" if abs_effect > 0 else "↓"
        print(f"  {metric}: {rel_effect:+.1%} {direction} (p={p_value:.3f}) {sig}")
        
        return {
            "vertical": vertical_name,
            "metric": metric,
            "actual_avg": actual_avg,
            "predicted_avg": predicted_avg,
            "absolute_effect": abs_effect,
            "relative_effect": rel_effect,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "p_value": p_value,
            "significant": p_value < 0.05,
            "ci_object": ci
        }
    except Exception as e:
        print(f"  {metric}: Error - {str(e)[:50]}")
        return None

if USE_CLUSTERING:
    print("\n" + "=" * 80)
    print("RUNNING CAUSAL IMPACT ANALYSIS BY CLUSTER")
    print("=" * 80)
    
    CLUSTER_METRICS = ["ivr", "cvr", "vvr", "cpa", "cpv", "roas", "aov"]
    MIN_CLUSTER_SIZE = 5  # Skip clusters with fewer than this many advertisers
    
    # Define covariates for causal impact (must match columns in cluster_weekly)
    COVARIATES = ["impressions", "spend", "uniques", "active_advertisers", "active_campaigns"]
    
    cluster_results = {}
    
    release_ts = pd.Timestamp(RELEASE_DATE)
    
    # Get cluster sizes from adv_features
    cluster_sizes = adv_features.groupby("cluster").size().to_dict()
    
    for cluster_id in sorted(cluster_weekly["cluster"].unique()):
        label = cluster_labels[cluster_id]
        n_advertisers = cluster_sizes.get(cluster_id, 0)
        
        print(f"\n{'='*70}")
        print(f"{label}")
        print(f"{'='*70}")
        
        # Skip tiny clusters
        if n_advertisers < MIN_CLUSTER_SIZE:
            print(f"⚠️ Only {n_advertisers} advertisers - skipping (min: {MIN_CLUSTER_SIZE})")
            continue
        
        cdata = cluster_weekly[cluster_weekly["cluster"] == cluster_id].copy()
        cdata = cdata.sort_values("week_start").set_index("week_start")
        
        n_pre = (cdata.index < release_ts).sum()
        n_post = (cdata.index >= release_ts).sum()
        
        print(f"Advertisers: {n_advertisers} | Pre-period: {n_pre} weeks | Post-period: {n_post} weeks")
        
        if n_pre < 12 or n_post < 2:
            print("⚠️ Insufficient data - skipping")
            continue
        
        pre_period = [cdata.index.min(), cdata.index[cdata.index < release_ts].max()]
        post_period = [cdata.index[cdata.index >= release_ts].min(), cdata.index.max()]
        
        cluster_results[cluster_id] = {}
        
        for metric in CLUSTER_METRICS:
            if cdata[metric].isna().all():
                print(f"  {metric}: All NaN - skipping")
                continue
            
            try:
                result = run_causal_impact_cluster(
                    data=cdata,
                    metric=metric,
                    covariates=COVARIATES,
                    pre_period=pre_period,
                    post_period=post_period,
                    vertical_name=label
                )
                if result:
                    cluster_results[cluster_id][metric] = result
                    
            except Exception as e:
                print(f"  {metric}: Error - {str(e)[:50]}")
    
    print("\n" + "=" * 80)
    print("CLUSTER-BASED CAUSAL IMPACT ANALYSIS COMPLETE")
    print("=" * 80)
    
    # Create summary table of cluster results
    if cluster_results:
        summary_rows = []
        for cid, metrics in cluster_results.items():
            for metric, result in metrics.items():
                summary_rows.append({
                    "cluster": cluster_labels.get(cid, f"C{cid}"),
                    "metric": metric,
                    "relative_effect_pct": f"{result['relative_effect']*100:.2f}%",
                    "p_value": f"{result['p_value']:.4f}",
                    "significant": "✓" if result['significant'] else "",
                    "direction": "↑" if result['relative_effect'] > 0 else "↓"
                })
        
        if summary_rows:
            cluster_summary_df = pd.DataFrame(summary_rows)
            print("\n" + "=" * 80)
            print("CLUSTER CAUSAL IMPACT SUMMARY")
            print("=" * 80)
            display(cluster_summary_df)
            
            # Show significant results only
            sig_results = cluster_summary_df[cluster_summary_df["significant"] == "✓"]
            if len(sig_results) > 0:
                print("\n" + "=" * 80)
                print("SIGNIFICANT RESULTS (p < 0.05)")
                print("=" * 80)
                display(sig_results)
            else:
                print("\nNo statistically significant results at p < 0.05")
    else:
        print("\nNo cluster results to summarize")

In [ ]:
# =============================================================================
# PLOT CLUSTER CAUSAL IMPACT RESULTS
# =============================================================================

def plot_cluster_causal_impact(result: dict, figsize=(14, 10)):
    """
    Plot CausalImpact results for a cluster.
    Y-axis is dynamically scaled based on actual/predicted values, not CI extremes.
    """
    if result is None or "ci_object" not in result:
        print(f"No CI object in result")
        return
    
    ci = result["ci_object"]
    cluster_name = result.get("vertical", "Unknown Cluster")
    metric = result.get("metric", "Unknown")
    rel_effect = result.get("relative_effect", 0)
    p_value = result.get("p_value", 1)
    significant = "✓ SIGNIFICANT" if p_value < 0.05 else ""
    
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)
    
    # Get data from causal impact object
    inferences = ci.inferences
    
    # Get actual observed values - try different approaches
    try:
        if hasattr(ci, 'data') and ci.data is not None:
            actual_values = ci.data.iloc[:, 0]
        elif hasattr(ci, 'pre_data') and hasattr(ci, 'post_data'):
            actual_values = pd.concat([ci.pre_data.iloc[:, 0], ci.post_data.iloc[:, 0]])
        else:
            actual_values = inferences["preds"] + inferences["point_effects"]
    except Exception as e:
        print(f"  Warning: Could not get actual values ({e}), using reconstructed")
        actual_values = inferences["preds"] + inferences["point_effects"]
    
    # Ensure index alignment
    actual_values = actual_values.reindex(inferences.index)
    
    # ==========================================================================
    # Panel 1: Actual vs Predicted
    # ==========================================================================
    ax1 = axes[0]
    ax1.plot(inferences.index, actual_values, label="Actual", color="blue", linewidth=2)
    ax1.plot(inferences.index, inferences["preds"], label="Predicted (Counterfactual)", 
             color="orange", linewidth=2, linestyle="--")
    ax1.fill_between(inferences.index, inferences["preds_lower"], inferences["preds_upper"], 
                     alpha=0.2, color="orange", label="95% CI")
    ax1.axvline(x=pd.Timestamp(RELEASE_DATE), color="red", linestyle="--", linewidth=2, label="Release Date")
    ax1.set_ylabel(metric.upper())
    ax1.legend(loc="upper left")
    ax1.set_title(f"Causal Impact: {metric.upper()} - {cluster_name}\n"
                  f"Relative Effect: {rel_effect:+.2%} (p={p_value:.4f}) {significant}", fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # FIX: Set y-axis limits based on actual/predicted data, not CI extremes
    y_data_1 = pd.concat([actual_values.dropna(), inferences["preds"].dropna()])
    y_min_1, y_max_1 = y_data_1.min(), y_data_1.max()
    y_range_1 = y_max_1 - y_min_1
    padding_1 = y_range_1 * 0.2 if y_range_1 > 0 else abs(y_max_1) * 0.2
    ax1.set_ylim(y_min_1 - padding_1, y_max_1 + padding_1)
    
    # ==========================================================================
    # Panel 2: Point Effect (difference between actual and predicted)
    # ==========================================================================
    ax2 = axes[1]
    ax2.plot(inferences.index, inferences["point_effects"], color="green", linewidth=2)
    ax2.fill_between(inferences.index, inferences["point_effects_lower"], inferences["point_effects_upper"],
                     alpha=0.2, color="green")
    ax2.axhline(y=0, color="black", linestyle="-", linewidth=1)
    ax2.axvline(x=pd.Timestamp(RELEASE_DATE), color="red", linestyle="--", linewidth=2)
    ax2.set_ylabel("Point Effect")
    ax2.set_title("Pointwise Causal Effect (Actual - Predicted)")
    ax2.grid(True, alpha=0.3)
    
    # FIX: Set y-axis limits based on point effects data, not CI extremes
    y_data_2 = inferences["point_effects"].dropna()
    y_min_2, y_max_2 = y_data_2.min(), y_data_2.max()
    y_range_2 = y_max_2 - y_min_2
    padding_2 = y_range_2 * 0.3 if y_range_2 > 0 else abs(y_max_2) * 0.3
    # Ensure 0 is visible
    ax2.set_ylim(min(y_min_2 - padding_2, -padding_2/2), max(y_max_2 + padding_2, padding_2/2))
    
    # ==========================================================================
    # Panel 3: Cumulative Effect (already scales well, but apply same logic)
    # ==========================================================================
    ax3 = axes[2]
    ax3.plot(inferences.index, inferences["post_cum_effects"], color="purple", linewidth=2)
    ax3.fill_between(inferences.index, inferences["post_cum_effects_lower"], inferences["post_cum_effects_upper"],
                     alpha=0.2, color="purple")
    ax3.axhline(y=0, color="black", linestyle="-", linewidth=1)
    ax3.axvline(x=pd.Timestamp(RELEASE_DATE), color="red", linestyle="--", linewidth=2)
    ax3.set_ylabel("Cumulative Effect")
    ax3.set_xlabel("Week")
    ax3.set_title("Cumulative Causal Effect Since Release")
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plot all cluster results
if USE_CLUSTERING and cluster_results:
    print("=" * 80)
    print("CLUSTER CAUSAL IMPACT VISUALIZATIONS")
    print("=" * 80)
    
    # Count total plots
    total_plots = sum(len(metrics) for metrics in cluster_results.values())
    print(f"Generating {total_plots} plots across {len(cluster_results)} clusters...\n")
    
    for cluster_id in sorted(cluster_results.keys()):
        metrics = cluster_results[cluster_id]
        cluster_name = cluster_labels.get(cluster_id, f"Cluster {cluster_id}")
        
        print(f"\n{'='*70}")
        print(f"{cluster_name}")
        print(f"{'='*70}")
        
        for metric_name, result in metrics.items():
            if result and result.get("ci_object"):
                plot_cluster_causal_impact(result)
else:
    print("No cluster results to plot. Run cluster analysis first.")


In [ ]:
# =============================================================================
# STEP 9: SUMMARY - CLUSTER VS VERTICAL COMPARISON
# =============================================================================

if USE_CLUSTERING:
    print("\n" + "=" * 80)
    print("CLUSTER-BASED ANALYSIS SUMMARY")
    print("=" * 80)
    
    print("\nCluster composition (advertisers per cluster):")
    for c in range(CHOSEN_K):
        n = len(adv_features[adv_features["cluster"] == c])
        verts = adv_features[adv_features["cluster"] == c]["vertical_name"].value_counts().head(3)
        print(f"\n{cluster_labels[c]} (n={n}):")
        for v, cnt in verts.items():
            print(f"    {v}: {cnt} advertisers")
    
    print("\n\n" + "=" * 80)
    print("KEY INSIGHT: Behavioral clustering groups advertisers by how they")
    print("respond to changes, not by arbitrary business categories.")
    print("=" * 80)
    print("""
    ADVANTAGES OF CLUSTERING:
    • Better counterfactual prediction (similar behavior = better synthetic control)
    • May reveal effects hidden within heterogeneous verticals
    • Groups "responders" vs "non-responders" regardless of industry
    
    DISADVANTAGES:
    • Less interpretable for business stakeholders ("Cluster 3" vs "Restaurants")
    • Clusters may shift over time (need to re-cluster periodically)
    • Requires sufficient advertiser-level data
    
    RECOMMENDATION:
    • Use clustering for deep-dive analysis and effect detection
    • Use verticals for stakeholder communication and reporting
    • Compare results between both approaches for robustness
    """)

In [ ]:
# =============================================================================
# EXPORT RESULTS (Optional)
# =============================================================================

# Uncomment to export cluster results to CSV
# if USE_CLUSTERING and cluster_results:
#     export_rows = []
#     for cluster_id, metrics in cluster_results.items():
#         for metric_name, result in metrics.items():
#             if result:
#                 row = {k: v for k, v in result.items() if k != 'ci_object'}
#                 row['cluster_id'] = cluster_id
#                 export_rows.append(row)
#     
#     export_df = pd.DataFrame(export_rows)
#     export_df.to_csv('/dbfs/tmp/cluster_causal_impact_results.csv', index=False)
#     print('Results exported to /dbfs/tmp/cluster_causal_impact_results.csv')
